# Take Home Assignment 4: Multi-Agent Building Maintenance System using CrewAI

# Setup

Dependecies

In [1]:
!pip install -q crewai crewai-tools litellm
!pip install -q transformers torch accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Import & API key setup

In [3]:
import os
import time
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["OPENAI_API_KEY"] = "NA"

from crewai import Agent, Task, Crew, Process
from crewai.llm import LLM

llm = LLM(
    model="huggingface/meta-llama/Llama-3.1-8B-Instruct:cerebras",  # ← provider:cerebras suffix
    api_key=os.environ["HF_TOKEN"],
    temperature=0.3,
    max_tokens=1024
)

print("✅ Model:", llm.model)
print("✅ Setup complete!")

✅ Model: huggingface/meta-llama/Llama-3.1-8B-Instruct:cerebras
✅ Setup complete!


# Agents

## HVAC Agent

In [4]:
hvac_agent = Agent(
    role="HVAC Systems Monitor",
    goal="Monitor temperature, ventilation, and air quality throughout the building and report anomalies that may indicate danger or discomfort.",
    backstory="""You are a certified HVAC specialist with 15 years of experience
    managing building environmental systems. You have deep knowledge of air quality
    standards, ventilation protocols, and how environmental factors interact with
    fire and medical emergencies. You provide precise sensor readings and
    actionable recommendations.""",
    llm=llm,
    verbose=True,        # prints agent's thinking process
    allow_delegation=False
)

## Fire System Agent

In [5]:
fire_agent = Agent(
    role="Fire Safety and Hazard Detection Specialist",
    goal="Detect fire hazards, trigger appropriate alarms, coordinate suppression systems, and communicate fire status to other emergency responders.",
    backstory="""You are a fire safety engineer trained in building fire detection
    systems, suppression protocols, and emergency evacuation procedures. You work
    closely with police for evacuation and with HVAC to cut ventilation during fires.
    You are calm under pressure and always prioritize human life over property.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

## Police Agent

In [6]:
police_agent = Agent(
    role="Building Security and Evacuation Coordinator",
    goal="Handle security breaches, manage building evacuation, control access points, and ensure occupant safety during emergencies.",
    backstory="""You are a senior security officer with training in emergency
    evacuation management, crowd control, and crisis coordination. You liaise
    between fire services, medical teams, and building management. You make
    quick decisions to protect lives and secure the premises during incidents.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

## Ambulance Agent

In [7]:
ambulance_agent = Agent(
    role="Emergency Medical Response Coordinator",
    goal="Assess medical emergencies within the building, dispatch appropriate medical resources, and provide first-response guidance to minimize casualties.",
    backstory="""You are a paramedic team leader with 10 years of experience
    in emergency medical response. You assess injuries, triage patients, coordinate
    with hospitals, and guide on-site staff through first aid procedures while your
    team is in transit. You work closely with police for safe access routes and
    with HVAC to ensure air quality is safe for victims.""",
    llm=llm,
    verbose=True,
    allow_delegation=False
)

# Tasks & Scenario's

## Scenario 1

In [8]:
# Scenario 1: Fire breaks out on Floor 3 + person collapses in lobby

scenario_1_context = """
INCIDENT REPORT - Building Alpha, 14:35 PM
- Smoke detector triggered on Floor 3, Zone B
- Temperature spike detected: 78°C near server room
- One person reported unconscious in the lobby (possible cardiac event)
- 120 occupants currently in the building
- Main stairwell A is accessible, B is blocked by smoke
"""

# Task 1: HVAC Assessment
task_hvac = Task(
    description=f"""
    Analyze the following building incident and provide a full HVAC assessment:
    {scenario_1_context}

    Your job:
    1. Report current air quality and temperature readings by floor
    2. Identify ventilation zones affected by the smoke
    3. Recommend whether to shut down or redirect HVAC systems
    4. Flag any zones where air quality is dangerous for occupants
    5. Provide recommendations for the fire and medical teams
    """,
    expected_output="""A structured HVAC status report including:
    - Temperature and air quality readings per floor
    - Ventilation shutdown recommendations
    - Zones safe vs unsafe for occupants
    - Specific guidance for fire and medical responders""",
    agent=hvac_agent
)

# Task 2: Fire Assessment
task_fire = Task(
    description=f"""
    Based on the incident report and the HVAC assessment provided:
    {scenario_1_context}

    Your job:
    1. Confirm fire hazard level and affected zones
    2. Trigger appropriate alarm levels for each floor
    3. Activate or recommend suppression systems
    4. Identify safe and unsafe evacuation routes
    5. Communicate critical information to police and medical teams
    """,
    expected_output="""A fire response report including:
    - Confirmed hazard zones and severity level
    - Alarm and suppression actions taken
    - Safe evacuation routes identified
    - Handoff instructions for police evacuation coordinator""",
    agent=fire_agent,
    context=[task_hvac]  # ← fire agent reads HVAC output
)

# Task 3: Police/Evacuation Response
task_police = Task(
    description=f"""
    Based on the fire report and HVAC assessment:
    {scenario_1_context}

    Your job:
    1. Initiate and coordinate full building evacuation
    2. Assign security personnel to each exit point
    3. Secure the building perimeter for emergency vehicles
    4. Account for all 120 occupants and identify vulnerable individuals
    5. Ensure ambulance has a clear access route to the lobby
    """,
    expected_output="""An evacuation coordination report including:
    - Evacuation routes assigned per floor
    - Personnel deployment plan
    - Perimeter security status
    - Clear access route confirmed for ambulance
    - Occupant headcount and status""",
    agent=police_agent,
    context=[task_hvac, task_fire]  # ← reads both previous outputs
)

# Task 4: Medical Response
task_ambulance = Task(
    description=f"""
    Based on all prior reports from HVAC, fire, and police teams:
    {scenario_1_context}

    Your job:
    1. Assess the unconscious person in the lobby
    2. Determine if any other occupants need medical attention (smoke inhalation etc.)
    3. Coordinate ambulance access route with police
    4. Provide first aid instructions to bystanders until paramedics arrive
    5. Arrange hospital notification and patient triage plan
    """,
    expected_output="""A medical response report including:
    - Patient assessment and priority triage
    - First aid instructions issued to bystanders
    - Ambulance access route confirmed
    - Hospital notification status
    - Full casualty count and condition summary""",
    agent=ambulance_agent,
    context=[task_hvac, task_fire, task_police]  # ← reads all previous outputs
)

print("✅ All tasks defined for Scenario 1!")

✅ All tasks defined for Scenario 1!


Scenario 1 Output

In [9]:
# ── Cell 5: Assemble & Run Scenario 1 ─────────────────────────

crew_scenario_1 = Crew(
    agents=[hvac_agent, fire_agent, police_agent, ambulance_agent],
    tasks=[task_hvac, task_fire, task_police, task_ambulance],
    process=Process.sequential,
    verbose=True
)

print("🚨 Starting Scenario 1: Fire + Medical Emergency...")
print("=" * 60)

result_scenario_1 = crew_scenario_1.kickoff()

print("=" * 60)
print("✅ Scenario 1 Complete!")
print("\n📋 FINAL COMBINED REPORT:")
print(result_scenario_1)

🚨 Starting Scenario 1: Fire + Medical Emergency...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: b37c26e9-34b0-4cc2-8afc-7356b73833ce                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analyze the following building incident and provide a full HVAC assessment:                                │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 14:35 PM                                                                     │
│  - Smoke detector triggered on Floor 3, Zone B                                                                  │
│  - Temperature spike detected: 78°C near server room                                                            │
│  - One person reported unconscious in the lobby (possible cardiac event)                                        │
│  - 120 occupants currently in the building                                                                      │
│  - Main stairwell A is accessible, B is blocked by smoke                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Report current air quality and temperature readings by floor                                            │
│      2. Identify ventilation zones affected by the smoke                                                        │
│      3. Recommend whether to shut down or redirect HVAC systems                                                 │
│      4. Flag any zones where air quality is dangerous for occupants                                             │
│      5. Provide recommendations for the fire and medical teams                                                  │
│                                                                                                                 │
│  ID: eb7c8cbc-7c1b-4cc9-a1ff-b2f17484c2b4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HVAC Systems Monitor                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analyze the following building incident and provide a full HVAC assessment:                                │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 14:35 PM                                                                     │
│  - Smoke detector triggered on Floor 3, Zone B                                                                  │
│  - Temperature spike detected: 78°C near server room                                                            │
│  - One person reported unconscious in the lobby (possible cardiac event)                                        │
│  - 120 occupants currently in the building                                                                      │
│  - Main stairwell A is accessible, B is blocked by smoke                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Report current air quality and temperature readings by floor                                            │
│      2. Identify ventilation zones affected by the smoke                                                        │
│      3. Recommend whether to shut down or redirect HVAC systems                                                 │
│      4. Flag any zones where air quality is dangerous for occupants                                             │
│      5. Provide recommendations for the fire and medical teams                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HVAC Systems Monitor                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **HVAC Status Report - Building Alpha**                                                                        │
│                                                                                                                 │
│  **Incident Time:** 14:35 PM                                                                                    │
│  **Occupant Count:** 120                                                                                        │
│  **Affected Areas:** Floor 3, Zone B; Lobby                                                                     │
│                                                                                                                 │
│  **Temperature and Air Quality Readings per Floor:**                                                            │
│                                                                                                                 │
│  | Floor | Temperature (°C) | Air Quality Index (AQI) |                                                         │
│  | --- | --- | --- |                                                                                            │
│  | 1 | 22.5 | 50 (Good) |                                                                                       │
│  | 2 | 23.2 | 40 (Good) |                                                                                       │
│  | 3 | 78.0 | 1000 (Hazardous) |                                                                                │
│  | 4 | 22.8 | 30 (Good) |                                                                                       │
│  | 5 | 23.5 | 20 (Good) |                                                                                       │
│                                                                                                                 │
│  **Ventilation Zones Affected by Smoke:**                                                                       │
│                                                                                                                 │
│  * Zone B (Floor 3): Smoke detected, ventilation system shutdown recommended                                    │
│  * Zone A (Floor 3): Potential for smoke infiltration, ventilation system monitoring recommended                │
│  * Lobby: Potential for smoke infiltration, ventilation system monitoring recommended                           │
│                                                                                                                 │
│  **Recommendations for HVAC Systems:**                                                                          │
│                                                                                                                 │
│  * Shut down ventilation systems serving Zone B (Floor 3) to prevent smoke spread                               │
│  * Redirect ventilation systems serving Zone A (Floor 3) to provide clean air from other zones                  │
│  * Maintain ventilation systems serving floors 1, 2, 4, and 5 to provide good air quality                       │
│                                                                                                                 │
│  **Zones Safe vs Unsafe for Occupants:**                                                                        │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Analyze the following building incident and provide a full HVAC assessment:                                │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 14:35 PM                                                                     │
│  - Smoke detector triggered on Floor 3, Zone B                                                                  │
│  - Temperature spike detected: 78°C near server room                                                            │
│  - One person reported unconscious in the lobby (possible cardiac event)                                        │
│  - 120 occupants currently in the building                                                                      │
│  - Main stairwell A is accessible, B is blocked by smoke                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Report current air quality and temperature readings by floor                                            │
│      2. Identify ventilation zones affected by the smoke                                                        │
│      3. Recommend whether to shut down or redirect HVAC systems                                                 │
│      4. Flag any zones where air quality is dangerous for occupants                                             │
│      5. Provide recommendations for the fire and medical teams                                                  │
│                                                                                                                 │
│  Agent: HVAC Systems Monitor                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (9 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Emergency Medical Response Coordinator                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Based on all prior reports from HVAC, fire, and police teams:                                              │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 14:35 PM                                                                     │
│  - Smoke detector triggered on Floor 3, Zone B                                                                  │
│  - Temperature spike detected: 78°C near server room                                                            │
│  - One person reported unconscious in the lobby (possible cardiac event)                                        │
│  - 120 occupants currently in the building                                                                      │
│  - Main stairwell A is accessible, B is blocked by smoke                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Assess the unconscious person in the lobby                                                              │
│      2. Determine if any other occupants need medical attention (smoke inhalation etc.)                         │
│      3. Coordinate ambulance access route with police                                                           │
│      4. Provide first aid instructions to bystanders until paramedics arrive                                    │
│      5. Arrange hospital notification and patient triage plan                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Emergency Medical Response Coordinator                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Medical Response Report - Building Alpha**                                                                   │
│                                                                                                                 │
│  **Incident Time:** 14:35 PM                                                                                    │
│  **Occupant Count:** 120                                                                                        │
│  **Affected Areas:** Floor 3, Zone B; Lobby                                                                     │
│                                                                                                                 │
│  **Patient Assessment and Priority Triage:**                                                                    │
│                                                                                                                 │
│  1. The unconscious individual in the lobby is assessed as a priority 1 patient, requiring immediate medical    │
│  attention. The patient is experiencing a possible cardiac event, and their airway, breathing, and circulation  │
│  (ABCs) are compromised. The patient is administered oxygen and cardiac medication to stabilize their           │
│  condition.                                                                                                     │
│  2. A thorough assessment of the patient's medical history and current condition is conducted, and a plan is    │
│  developed for further medical treatment and transportation to the hospital.                                    │
│  3. The patient is transported to the hospital via ambulance, with a paramedic team providing ongoing medical   │
│  care and monitoring during transport.                                                                          │
│                                                                                                                 │
│  **First Aid Instructions Issued to Bystanders:**                                                               │
│                                                                                                                 │
│  1. Bystanders are instructed to remain calm and provide basic first aid to any injured individuals, including  │
│  the unconscious patient in the lobby.                                                                          │
│  2. Bystanders are advised to call 911 or the emergency services number in their area to report the incident    │
│  and request medical assistance.                                                                                │
│  3. Bystanders are instructed to provide basic first aid, including CPR and oxygen administration, until        │
│  medical help arrives.                                                                                          │
│                                                                                                                 │
│  **Ambulance Access Route Confirmed:**                                                                          │
│                                                                                                                 │
│  1. A clear access route has been confirmed for the ambulance to reach the lobby, with security personnel       │
│  securing the route to ensure safe passage for medical 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Based on all prior reports from HVAC, fire, and police teams:                                              │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 14:35 PM                                                                     │
│  - Smoke detector triggered on Floor 3, Zone B                                                                  │
│  - Temperature spike detected: 78°C near server room                                                            │
│  - One person reported unconscious in the lobby (possible cardiac event)                                        │
│  - 120 occupants currently in the building                                                                      │
│  - Main stairwell A is accessible, B is blocked by smoke                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Assess the unconscious person in the lobby                                                              │
│      2. Determine if any other occupants need medical attention (smoke inhalation etc.)                         │
│      3. Coordinate ambulance access route with police                                                           │
│      4. Provide first aid instructions to bystanders until paramedics arrive                                    │
│      5. Arrange hospital notification and patient triage plan                                                   │
│                                                                                                                 │
│  Agent: Emergency Medical Response Coordinator                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Scenario 1 Complete!

📋 FINAL COMBINED REPORT:
**Medical Response Report - Building Alpha**

**Incident Time:** 14:35 PM
**Occupant Count:** 120
**Affected Areas:** Floor 3, Zone B; Lobby

**Patient Assessment and Priority Triage:**

1. The unconscious individual in the lobby is assessed as a priority 1 patient, requiring immediate medical attention. The patient is experiencing a possible cardiac event, and their airway, breathing, and circulation (ABCs) are compromised. The patient is administered oxygen and cardiac medication to stabilize their condition.
2. A thorough assessment of the patient's medical history and current condition is conducted, and a plan is developed for further medical treatment and transportation to the hospital.
3. The patient is transported to the hospital via ambulance, with a paramedic team providing ongoing medical care and monitoring during transport.

**First Aid Instructions Issued to Bystanders:**

1. Bystanders are instructed to remain calm and prov

In [18]:
with open("scenario_1_output.txt", "w") as f:
    f.write("SCENARIO 1: Fire + Medical Emergency\n")
    f.write("=" * 60 + "\n\n")
    f.write(str(result_scenario_1))

    print("✅ Scenario 1 output saved!")

✅ Scenario 1 output saved!


## Scenario 2

In [10]:
# Cell 6: Define Tasks for Scenario 2
# Scenario 2: Security breach + Air quality failure (gas leak)

scenario_2_context = """
INCIDENT REPORT - Building Alpha, 09:15 AM
- Unauthorized access detected at Server Room B2, basement level
- Security camera footage shows 2 unidentified individuals
- CO2 levels spiking abnormally on Floors 1 and 2 (possible gas leak)
- Air quality sensors reading HAZARDOUS on Floor 2, Zone A
- 85 occupants currently in the building (morning shift)
- One security guard reported feeling dizzy and nauseous near Floor 2
- Emergency exit on east wing is jammed
"""

# Task 1: HVAC Assessment
task2_hvac = Task(
    description=f"""
    Analyze the following building incident and provide a full HVAC assessment:
    {scenario_2_context}

    Your job:
    1. Report CO2 and air quality readings per floor
    2. Identify the source and spread pattern of the gas leak
    3. Recommend immediate ventilation actions to contain the leak
    4. Flag which zones must be evacuated immediately
    5. Advise fire and medical teams on respiratory risks
    """,
    expected_output="""A structured HVAC status report including:
    - CO2 and air quality readings per floor
    - Gas leak source identification and spread pattern
    - Ventilation actions recommended
    - Zones requiring immediate evacuation
    - Respiratory risk advisory for other teams""",
    agent=hvac_agent
)

# Task 2: Fire Assessment
task2_fire = Task(
    description=f"""
    Based on the incident report and HVAC assessment:
    {scenario_2_context}

    Your job:
    1. Assess whether the gas leak poses an explosion or fire risk
    2. Determine if suppression systems should be activated or held
    3. Identify all safe and blocked evacuation routes
    4. Coordinate with police on whether to lock down or evacuate
    5. Issue fire hazard level and standing orders to all teams
    """,
    expected_output="""A fire hazard assessment including:
    - Explosion/fire risk level from gas leak
    - Suppression system recommendation (activate or hold)
    - Safe and blocked evacuation routes
    - Lockdown vs evacuation recommendation
    - Standing orders issued to all teams""",
    agent=fire_agent,
    context=[task2_hvac]
)

# Task 3: Police Response
task2_police = Task(
    description=f"""
    Based on the fire and HVAC reports:
    {scenario_2_context}

    Your job:
    1. Respond to the security breach in Server Room B2
    2. Identify and apprehend or track the unauthorized individuals
    3. Coordinate evacuation given the jammed east wing exit
    4. Secure the building perimeter and lock down access points
    5. Ensure medical team has safe access to the dizzy security guard
    """,
    expected_output="""A security and evacuation report including:
    - Security breach response actions taken
    - Status of unauthorized individuals
    - Evacuation plan around jammed exit
    - Perimeter lockdown status
    - Safe access route confirmed for medical team""",
    agent=police_agent,
    context=[task2_hvac, task2_fire]
)

# Task 4: Medical Response
task2_ambulance = Task(
    description=f"""
    Based on all prior reports from HVAC, fire, and police teams:
    {scenario_2_context}

    Your job:
    1. Assess the security guard with dizziness and nausea (possible gas inhalation)
    2. Screen other occupants on Floors 1 and 2 for symptoms
    3. Establish a safe medical triage point away from affected zones
    4. Provide first aid guidance for gas inhalation to bystanders
    5. Coordinate with hospital for potential mass casualty scenario
    """,
    expected_output="""A medical response report including:
    - Security guard assessment and treatment plan
    - Occupant symptom screening results
    - Triage point location established
    - First aid instructions for gas inhalation issued
    - Hospital notification and mass casualty preparedness status""",
    agent=ambulance_agent,
    context=[task2_hvac, task2_fire, task2_police]
)

print("✅ All tasks defined for Scenario 2!")

✅ All tasks defined for Scenario 2!


Scenario 2 Output

In [12]:
crew_scenario_2 = Crew(
    agents=[hvac_agent, fire_agent, police_agent, ambulance_agent],
    tasks=[task2_hvac, task2_fire, task2_police, task2_ambulance],
    process=Process.sequential,
    verbose=True
)

print("🚨 Starting Scenario 2: Security Breach + Gas Leak...")
print("=" * 60)

result_scenario_2 = crew_scenario_2.kickoff()

print("=" * 60)
print("✅ Scenario 2 Complete!")
print("\n📋 FINAL COMBINED REPORT:")
print(result_scenario_2)

🚨 Starting Scenario 2: Security Breach + Gas Leak...


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7bd28242-ba44-414a-be88-c761e80d0a5d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analyze the following building incident and provide a full HVAC assessment:                                │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 09:15 AM                                                                     │
│  - Unauthorized access detected at Server Room B2, basement level                                               │
│  - Security camera footage shows 2 unidentified individuals                                                     │
│  - CO2 levels spiking abnormally on Floors 1 and 2 (possible gas leak)                                          │
│  - Air quality sensors reading HAZARDOUS on Floor 2, Zone A                                                     │
│  - 85 occupants currently in the building (morning shift)                                                       │
│  - One security guard reported feeling dizzy and nauseous near Floor 2                                          │
│  - Emergency exit on east wing is jammed                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Report CO2 and air quality readings per floor                                                           │
│      2. Identify the source and spread pattern of the gas leak                                                  │
│      3. Recommend immediate ventilation actions to contain the leak                                             │
│      4. Flag which zones must be evacuated immediately                                                          │
│      5. Advise fire and medical teams on respiratory risks                                                      │
│                                                                                                                 │
│  ID: 3359d5f7-7134-4352-a0bd-73b33ed77bd0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HVAC Systems Monitor                                                                                    │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analyze the following building incident and provide a full HVAC assessment:                                │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 09:15 AM                                                                     │
│  - Unauthorized access detected at Server Room B2, basement level                                               │
│  - Security camera footage shows 2 unidentified individuals                                                     │
│  - CO2 levels spiking abnormally on Floors 1 and 2 (possible gas leak)                                          │
│  - Air quality sensors reading HAZARDOUS on Floor 2, Zone A                                                     │
│  - 85 occupants currently in the building (morning shift)                                                       │
│  - One security guard reported feeling dizzy and nauseous near Floor 2                                          │
│  - Emergency exit on east wing is jammed                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Report CO2 and air quality readings per floor                                                           │
│      2. Identify the source and spread pattern of the gas leak                                                  │
│      3. Recommend immediate ventilation actions to contain the leak                                             │
│      4. Flag which zones must be evacuated immediately                                                          │
│      5. Advise fire and medical teams on respiratory risks                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: HVAC Systems Monitor                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **HVAC Status Report - Building Alpha**                                                                        │
│                                                                                                                 │
│  **Incident Time:** 09:15 AM                                                                                    │
│  **Occupant Count:** 85                                                                                         │
│  **Affected Areas:** Server Room B2, basement level; Floors 1 and 2                                             │
│                                                                                                                 │
│  **CO2 and Air Quality Readings per Floor:**                                                                    │
│                                                                                                                 │
│  | Floor | CO2 (ppm) | Air Quality Index (AQI) |                                                                │
│  | --- | --- | --- |                                                                                            │
│  | 1 | 1500 | 800 (Hazardous) |                                                                                 │
│  | 2 | 2500 | 1000 (Hazardous) |                                                                                │
│  | 3 | 400 | 20 (Good) |                                                                                        │
│  | 4 | 300 | 10 (Good) |                                                                                        │
│  | 5 | 200 | 5 (Good) |                                                                                         │
│                                                                                                                 │
│  **Source and Spread Pattern of the Gas Leak:**                                                                 │
│                                                                                                                 │
│  Based on CO2 and air quality sensor readings, it appears that the gas leak is originating from Server Room     │
│  B2, basement level. The high CO2 levels and hazardous air quality readings on Floors 1 and 2 indicate that     │
│  the gas is spreading through the ventilation system. The spread pattern appears to be primarily through the    │
│  east wing, with some potential for lateral spread through the basement level.                                  │
│                                                                                                                 │
│  **Recommendations for Immediate Ventilation Actions:**                                                         │
│                                                                                                                 │
│  1. **Isolate the affected areas:** Shut down ventilation systems serving Floors 1 and 2 to prevent further     │
│  spread of the gas.                                                                                             │
│  2. **Contain the leak:** Activate the building's emergency ventilation system to create a negative pressure    │
│  in the affected areas, which will help to contain the gas leak.                                                │
│  3. **Provide clean air:** Redirect ventilation systems

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Analyze the following building incident and provide a full HVAC assessment:                                │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 09:15 AM                                                                     │
│  - Unauthorized access detected at Server Room B2, basement level                                               │
│  - Security camera footage shows 2 unidentified individuals                                                     │
│  - CO2 levels spiking abnormally on Floors 1 and 2 (possible gas leak)                                          │
│  - Air quality sensors reading HAZARDOUS on Floor 2, Zone A                                                     │
│  - 85 occupants currently in the building (morning shift)                                                       │
│  - One security guard reported feeling dizzy and nauseous near Floor 2                                          │
│  - Emergency exit on east wing is jammed                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Report CO2 and air quality readings per floor                                                           │
│      2. Identify the source and spread pattern of the gas leak                                                  │
│      3. Recommend immediate ventilation actions to contain the leak                                             │
│      4. Flag which zones must be evacuated immediately                                                          │
│      5. Advise fire and medical teams on respiratory risks                                                      │
│                                                                                                                 │
│  Agent: HVAC Systems Monitor                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (9 outputs left out)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Emergency Medical Response Coordinator                                                                  │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Based on all prior reports from HVAC, fire, and police teams:                                              │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 09:15 AM                                                                     │
│  - Unauthorized access detected at Server Room B2, basement level                                               │
│  - Security camera footage shows 2 unidentified individuals                                                     │
│  - CO2 levels spiking abnormally on Floors 1 and 2 (possible gas leak)                                          │
│  - Air quality sensors reading HAZARDOUS on Floor 2, Zone A                                                     │
│  - 85 occupants currently in the building (morning shift)                                                       │
│  - One security guard reported feeling dizzy and nauseous near Floor 2                                          │
│  - Emergency exit on east wing is jammed                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Assess the security guard with dizziness and nausea (possible gas inhalation)                           │
│      2. Screen other occupants on Floors 1 and 2 for symptoms                                                   │
│      3. Establish a safe medical triage point away from affected zones                                          │
│      4. Provide first aid guidance for gas inhalation to bystanders                                             │
│      5. Coordinate with hospital for potential mass casualty scenario                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Emergency Medical Response Coordinator                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Medical Response Report - Building Alpha**                                                                   │
│                                                                                                                 │
│  **Incident Time:** 09:15 AM                                                                                    │
│  **Occupant Count:** 85                                                                                         │
│  **Affected Areas:** Server Room B2, basement level; Floors 1 and 2                                             │
│                                                                                                                 │
│  **Security Guard Assessment and Treatment Plan:**                                                              │
│                                                                                                                 │
│  1. The security guard who is feeling dizzy and nauseous is assessed as a priority 1 patient, requiring         │
│  immediate medical attention.                                                                                   │
│  2. The patient is administered oxygen and cardiac medication to stabilize their condition.                     │
│  3. A thorough assessment of the patient's medical history and current condition is conducted, and a plan is    │
│  developed for further medical treatment and transportation to the hospital.                                    │
│  4. The patient is transported to the hospital via ambulance, with a paramedic team providing ongoing medical   │
│  care and monitoring during transport.                                                                          │
│                                                                                                                 │
│  **Occupant Symptom Screening Results:**                                                                        │
│                                                                                                                 │
│  1. A thorough screening of occupants on Floors 1 and 2 is conducted to identify any individuals who are        │
│  experiencing symptoms of gas inhalation.                                                                       │
│  2. A total of 10 occupants are identified as experiencing symptoms, including dizziness, nausea, and           │
│  shortness of breath.                                                                                           │
│  3. The occupants are administered oxygen and cardiac medication to stabilize their condition, and a plan is    │
│  developed for further medical treatment and transportation to the hospital.                                    │
│  4. The occupants are transported to the hospital via ambulance, with a paramedic team providing ongoing        │
│  medical care and monitoring during transport.                                                                  │
│                                                                                                                 │
│  **Triage Point Location Established:**                                                                         │
│                                                                                                                 │
│  1. A safe and secure triage point is established on th

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Based on all prior reports from HVAC, fire, and police teams:                                              │
│                                                                                                                 │
│  INCIDENT REPORT - Building Alpha, 09:15 AM                                                                     │
│  - Unauthorized access detected at Server Room B2, basement level                                               │
│  - Security camera footage shows 2 unidentified individuals                                                     │
│  - CO2 levels spiking abnormally on Floors 1 and 2 (possible gas leak)                                          │
│  - Air quality sensors reading HAZARDOUS on Floor 2, Zone A                                                     │
│  - 85 occupants currently in the building (morning shift)                                                       │
│  - One security guard reported feeling dizzy and nauseous near Floor 2                                          │
│  - Emergency exit on east wing is jammed                                                                        │
│                                                                                                                 │
│                                                                                                                 │
│      Your job:                                                                                                  │
│      1. Assess the security guard with dizziness and nausea (possible gas inhalation)                           │
│      2. Screen other occupants on Floors 1 and 2 for symptoms                                                   │
│      3. Establish a safe medical triage point away from affected zones                                          │
│      4. Provide first aid guidance for gas inhalation to bystanders                                             │
│      5. Coordinate with hospital for potential mass casualty scenario                                           │
│                                                                                                                 │
│  Agent: Emergency Medical Response Coordinator                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Scenario 2 Complete!

📋 FINAL COMBINED REPORT:
**Medical Response Report - Building Alpha**

**Incident Time:** 09:15 AM
**Occupant Count:** 85
**Affected Areas:** Server Room B2, basement level; Floors 1 and 2

**Security Guard Assessment and Treatment Plan:**

1. The security guard who is feeling dizzy and nauseous is assessed as a priority 1 patient, requiring immediate medical attention.
2. The patient is administered oxygen and cardiac medication to stabilize their condition.
3. A thorough assessment of the patient's medical history and current condition is conducted, and a plan is developed for further medical treatment and transportation to the hospital.
4. The patient is transported to the hospital via ambulance, with a paramedic team providing ongoing medical care and monitoring during transport.

**Occupant Symptom Screening Results:**

1. A thorough screening of occupants on Floors 1 and 2 is conducted to identify any individuals who are experiencing symptoms of gas inhala

In [19]:
with open("scenario_2_output.txt", "w") as f:
    f.write("SCENARIO 2: Security Breach + Gas Leak\n")
    f.write("=" * 60 + "\n\n")
    f.write(str(result_scenario_2))

print("✅ Scenario 2 output saved!")

✅ Scenario 2 output saved!


# System Design Document

In [15]:
# ── Cell 8: Save System Design Document ───────────────────────

system_design = """
================================================================
SYSTEM DESIGN DOCUMENT
Intelligent Building Maintenance & Emergency Response System
Using CrewAI Multi-Agent Framework
================================================================

1. INTRODUCTION
---------------
This document describes the design and implementation of an
intelligent multi-agent building maintenance and emergency
response system built using the CrewAI framework. The system
coordinates four specialized AI agents to detect, assess, and
respond to building emergencies in a realistic and structured
manner.

The system was designed to simulate real-world emergency
coordination where multiple teams must share information,
prioritize tasks, and make decisions based on each other's
findings — all within a unified automated pipeline.

----------------------------------------------------------------

2. SYSTEM ARCHITECTURE
-----------------------
The system follows a SEQUENTIAL MULTI-AGENT PIPELINE, where
each agent processes incoming incident data, generates a
structured report, and passes its output as context to the
next agent in the chain.

  [ Incident Report ]
         |
         v
  [ HVAC Agent ]
  - Reads raw sensor data
  - Outputs: air quality report
         |
         v
  [ Fire Agent ]
  - Reads HVAC report + incident
  - Outputs: hazard assessment + evacuation routes
         |
         v
  [ Police Agent ]
  - Reads HVAC + Fire reports
  - Outputs: evacuation plan + security response
         |
         v
  [ Ambulance Agent ]
  - Reads all 3 prior reports
  - Outputs: medical response + triage plan
         |
         v
  [ Final Combined Report ]

Framework Used  : CrewAI
LLM Provider    : Hugging Face (meta-llama/Llama-3.1-8B via Cerebras)
Process Type    : Sequential
Communication   : Context passing via CrewAI task context[]

----------------------------------------------------------------

3. AGENT DEFINITIONS
---------------------

3.1 HVAC Agent
  Role     : HVAC Systems Monitor
  Goal     : Monitor temperature, ventilation, and air quality
             and report anomalies that may indicate danger
  Backstory: Certified HVAC specialist with 15 years experience
             in building environmental systems
  Responsibilities:
    - Report temperature and air quality per floor
    - Identify ventilation zones affected by hazards
    - Recommend HVAC shutdown or redirection
    - Flag dangerous zones for other teams

3.2 Fire Systems Agent
  Role     : Fire Safety and Hazard Detection Specialist
  Goal     : Detect fire hazards, trigger alarms, coordinate
             suppression, communicate fire status
  Backstory: Fire safety engineer trained in detection systems,
             suppression protocols, and evacuation procedures
  Responsibilities:
    - Assess fire/explosion hazard level
    - Trigger alarms and activate suppression systems
    - Identify safe and blocked evacuation routes
    - Issue standing orders to all response teams

3.3 Police Agent
  Role     : Building Security and Evacuation Coordinator
  Goal     : Handle security breaches, manage evacuation,
             control access points, ensure occupant safety
  Backstory: Senior security officer with training in emergency
             evacuation and crisis coordination
  Responsibilities:
    - Coordinate full building evacuation
    - Respond to security breaches
    - Secure perimeter for emergency vehicles
    - Account for all occupants

3.4 Ambulance Agent
  Role     : Emergency Medical Response Coordinator
  Goal     : Assess medical emergencies, dispatch resources,
             provide first-response guidance
  Backstory: Paramedic team leader with 10 years experience in
             emergency medical response
  Responsibilities:
    - Triage and assess patients
    - Coordinate ambulance access routes
    - Issue first aid instructions to bystanders
    - Notify hospitals of incoming patients

----------------------------------------------------------------

4. INTER-AGENT COMMUNICATION
-----------------------------
Inter-agent communication is implemented using CrewAI's native
task context passing mechanism.

Each task receives the outputs of all previously completed
tasks as context. This simulates realistic information sharing
between emergency response teams:

  task_fire    = Task(context=[task_hvac])
  task_police  = Task(context=[task_hvac, task_fire])
  task_ambulance = Task(context=[task_hvac, task_fire, task_police])

This ensures that:
  - The Fire Agent builds on the HVAC Agent's findings
  - The Police Agent coordinates based on both HVAC and Fire reports
  - The Ambulance Agent has full situational awareness from
    all three prior agents before making medical decisions

----------------------------------------------------------------

5. SCENARIO DESCRIPTIONS
--------------------------

SCENARIO 1: Fire Emergency + Medical Emergency
  Location : Building Alpha
  Time     : 14:35 PM
  Triggers :
    - Smoke detector triggered on Floor 3, Zone B
    - Temperature spike of 78 degrees C near server room
    - Person unconscious in lobby (possible cardiac event)
    - 120 occupants in building
    - Main stairwell B blocked by smoke

  Agent Responses:
    HVAC     -> Flagged Floor 3 as unsafe, recommended
                ventilation shutdown for Zone B
    Fire     -> Confirmed high hazard, activated suppression,
                identified stairwell A as safe evacuation route
    Police   -> Initiated full evacuation, secured ambulance
                access route, deployed personnel to exits
    Ambulance-> Triaged cardiac patient, initiated CPR guidance,
                notified hospital, confirmed ambulance route

SCENARIO 2: Security Breach + Gas Leak
  Location : Building Alpha
  Time     : 09:15 AM
  Triggers :
    - Unauthorized access in Server Room B2 (basement)
    - 2 unidentified individuals on camera
    - CO2 spiking on Floors 1 and 2 (possible gas leak)
    - Air quality HAZARDOUS on Floor 2, Zone A
    - Security guard dizzy/nauseous near Floor 2
    - East wing emergency exit jammed
    - 85 occupants in building (morning shift)

  Agent Responses:
    HVAC     -> Identified gas leak source in basement,
                recommended ventilation isolation for Floors 1-2
    Fire     -> Assessed HIGH explosion risk, recommended
                evacuation over lockdown, identified west wing
                and stairwells A/B as safe routes
    Police   -> Tracked unauthorized individuals, coordinated
                evacuation around jammed east exit, secured
                perimeter, confirmed medical access route
    Ambulance-> Treated 11 symptomatic occupants, established
                triage point in west wing, notified hospital
                for mass casualty preparedness

----------------------------------------------------------------

6. DESIGN DECISIONS
--------------------

Sequential vs Hierarchical Process:
  Sequential process was chosen because emergency response
  follows a natural chain of dependency — medical teams cannot
  act without knowing safe access routes from police, police
  cannot evacuate without knowing which routes are safe from
  fire, and fire cannot respond without HVAC environmental data.

Single LLM for All Agents:
  All agents share the same underlying LLM. Each agent is
  differentiated by its role, goal, and backstory, which
  guides the LLM to respond from that agent's perspective.

allow_delegation = False:
  Delegation was disabled to keep agent responsibilities clear
  and prevent agents from reassigning tasks. Coordination is
  handled through context passing instead.

----------------------------------------------------------------

7. TECHNOLOGIES USED
---------------------
  - Python 3.12
  - CrewAI (multi-agent orchestration)
  - LiteLLM (LLM provider abstraction)
  - HuggingFace Inference API (LLM backend)
  - Meta Llama 3.1 8B Instruct via Cerebras provider
  - Google Colab (execution environment)

================================================================
"""
# Save to file
with open("system_design_document.txt", "w") as f:
    f.write(system_design)

print(system_design)
print("\n✅ System Design Document saved to system_design_document.txt")


SYSTEM DESIGN DOCUMENT
Intelligent Building Maintenance & Emergency Response System
Using CrewAI Multi-Agent Framework

1. INTRODUCTION
---------------
This document describes the design and implementation of an
intelligent multi-agent building maintenance and emergency
response system built using the CrewAI framework. The system
coordinates four specialized AI agents to detect, assess, and
respond to building emergencies in a realistic and structured
manner.

The system was designed to simulate real-world emergency
coordination where multiple teams must share information,
prioritize tasks, and make decisions based on each other's
findings — all within a unified automated pipeline.

----------------------------------------------------------------

2. SYSTEM ARCHITECTURE
-----------------------
The system follows a SEQUENTIAL MULTI-AGENT PIPELINE, where
each agent processes incoming incident data, generates a
structured report, and passes its output as context to the
next agent in the 

# Reflection Report

In [16]:
# ── Cell 9: Save Reflection Report ────────────────────────────

reflection_report = """
================================================================
REFLECTION REPORT
Intelligent Building Maintenance & Emergency Response System
================================================================

OVERVIEW
---------
This project involved designing and implementing a multi-agent
system using the CrewAI framework to simulate intelligent
building emergency response. Four specialized agents — HVAC,
Fire, Police, and Ambulance — were orchestrated to respond to
two distinct emergency scenarios in a coordinated, sequential
manner.

WHAT WORKED WELL
-----------------
The inter-agent communication mechanism worked effectively.
By passing each agent's output as context to the next agent,
the system achieved realistic coordination behavior. For
example, in Scenario 1, the Ambulance Agent's access route
decision was directly informed by the Police Agent's
evacuation plan, which itself was shaped by the Fire Agent's
hazard assessment. This chain of reasoning closely mirrors
how real emergency response teams operate.

The use of detailed backstories for each agent significantly
improved the quality and realism of outputs. Giving each
agent a specific professional background guided the LLM to
produce structured, domain-specific reports rather than
generic responses.

CHALLENGES FACED
-----------------
The main challenge was designing realistic scenarios with
enough complexity to require genuine coordination between
agents. Each scenario needed to involve multiple overlapping
emergencies so that each agent's output would meaningfully
affect the next agent's decisions.

LEARNINGS
----------
This project provided practical experience with key concepts
in multi-agent AI systems:

  1. Agent Design: Well-crafted roles, goals, and backstories
     are essential. Specific professional personas produce
     actionable, realistic outputs compared to vague
     agent definitions.

  2. Context Management: CrewAI's context passing is a simple
     yet powerful way to implement inter-agent communication
     without complex infrastructure.

  3. Sequential Orchestration: For workflows with clear
     dependency chains, sequential orchestration is the most
     logical and predictable approach.

CONCLUSION
-----------
The system successfully demonstrated the core principles of
multi-agent coordination — specialized agents, structured
communication, and emergent collaborative behavior. The two
scenarios showed how agents with different expertise can work
together to produce a comprehensive, unified emergency
response that no single agent could achieve alone.
================================================================
"""

with open("reflection_report.txt", "w") as f:
    f.write(reflection_report)

print(reflection_report)
print("\n✅ Reflection Report saved to reflection_report.txt")


REFLECTION REPORT
Intelligent Building Maintenance & Emergency Response System

OVERVIEW
---------
This project involved designing and implementing a multi-agent
system using the CrewAI framework to simulate intelligent
building emergency response. Four specialized agents — HVAC,
Fire, Police, and Ambulance — were orchestrated to respond to
two distinct emergency scenarios in a coordinated, sequential
manner.

WHAT WORKED WELL
-----------------
The inter-agent communication mechanism worked effectively.
By passing each agent's output as context to the next agent,
the system achieved realistic coordination behavior. For
example, in Scenario 1, the Ambulance Agent's access route
decision was directly informed by the Police Agent's
evacuation plan, which itself was shaped by the Fire Agent's
hazard assessment. This chain of reasoning closely mirrors
how real emergency response teams operate.

The use of detailed backstories for each agent significantly
improved the quality and realism of